In [1]:
import pandas as pd
import numpy as np

# preprocessing
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder
from imblearn.under_sampling import RandomUnderSampler

# model machine learning
from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,log_loss
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [2]:
df = pd.read_csv('../0.dataset/dataset_Heart_attack_clean.csv')
df_x = df.drop(columns='heart_attack')
df_y = df['heart_attack']

# 1. Binary Classification Random Forest

In [3]:
feature_categori = ['gender']

preprocessor = ColumnTransformer(
    transformers=[
        ('categori_encoding',OneHotEncoder(),feature_categori)
        ],
   remainder='passthrough' # Kolom kategori yang sudah ter-encode dari sananya akan aman lewat lewat sini
)

In [4]:
selector_model = RandomForestClassifier(random_state=42)
model_RandomForestClassifier = Pipeline([
    ('preprocessing',preprocessor),
    ('undersampling',RandomUnderSampler(random_state=42)),
    ('feature_selection',SequentialFeatureSelector(estimator=selector_model,direction='forward')),
    ('model_classifier',RandomForestClassifier(random_state=42))
])

In [5]:
params = {
    'feature_selection__n_features_to_select': [5],
    'model_classifier__n_estimators': [int(x) for x in np.linspace(start=20, stop=40, num=5)],
    'model_classifier__criterion': ['gini', 'entropy'],
    'model_classifier__max_depth': [7, 10, 12, 15],
    'model_classifier__min_samples_split': [24, 25, 30],
    'model_classifier__min_samples_leaf': [20, 25, 30],
    'model_classifier__max_features': ['sqrt', 'log2'],
    'model_classifier__bootstrap': [True],
    'model_classifier__class_weight': ['balanced', None]
}
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42,stratify=df_y)

random_search = RandomizedSearchCV(estimator=model_RandomForestClassifier,param_distributions=params,n_iter=2,cv=5,scoring='accuracy',random_state=42,n_jobs=-1,verbose=3)
random_search.fit(X_train,y_train)

Fitting 5 folds for each of 2 candidates, totalling 10 fits


,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'feature_selection__n_features_to_select': [5], 'model_classifier__bootstrap': [True], 'model_classifier__class_weight': ['balanced', None], 'model_classifier__criterion': ['gini', 'entropy'], ...}"
,n_iter,2
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [6]:
best_model_pipeline = random_search.best_estimator_
preprocessor_step = best_model_pipeline.named_steps['preprocessing']
sfs_step = best_model_pipeline.named_steps['feature_selection']

kolom_setelah_transformasi = preprocessor_step.get_feature_names_out()
fitur_terpilih = kolom_setelah_transformasi[sfs_step.get_support()]

print(f'\nHyperparameter Terbaik:\n{random_search.best_params_}')
print(f'\nFitur Terbaik Yang Terpilih:\n{list(fitur_terpilih)}')

#cek score
tes_accuracy = best_model_pipeline.score(X_test, y_test)

y_pred_test = best_model_pipeline.predict(X_test)
y_prob_test = best_model_pipeline.predict_proba(X_test)

y_pred_train = best_model_pipeline.predict(X_train)
y_prob_train = best_model_pipeline.predict_proba(X_train)

print(f'\nAkurasi Validasi Terbaik: {random_search.best_score_*100:.2f}%')
print(f'\nAkurasi Data Test: {tes_accuracy*100:.2f}%')


Hyperparameter Terbaik:
{'model_classifier__n_estimators': 20, 'model_classifier__min_samples_split': 25, 'model_classifier__min_samples_leaf': 20, 'model_classifier__max_features': 'log2', 'model_classifier__max_depth': 10, 'model_classifier__criterion': 'gini', 'model_classifier__class_weight': None, 'model_classifier__bootstrap': True, 'feature_selection__n_features_to_select': 5}

Fitur Terbaik Yang Terpilih:
['remainder__age', 'remainder__atrial_fibrillation', 'remainder__rheumatoid_arthritis', 'remainder__diabetes', 'remainder__time_to_event_or_censoring']

Akurasi Validasi Terbaik: 65.02%

Akurasi Data Test: 62.05%


In [7]:
def evaluate_model(pred,test,prob,evaluate,model_name='Decision Tree'):
    accuracy = accuracy_score(test,pred)
    precision = precision_score(test,pred)
    recall = recall_score(test,pred)
    f1 = f1_score(test,pred)
    roc_auc = roc_auc_score(test,prob[:,1])
    logloss = log_loss(test,prob)

    data = {
        'Model': [model_name],
        'Evaluated': [evaluate],
        'Accuracy': [accuracy],
        'Precision': [precision],
        'Recall': [recall],
        'F1-Score': [f1],
        'ROC-AUC': [roc_auc],
        'Log Loss': [logloss]
    }

    df_result = pd.DataFrame(data)
    return df_result

In [ ]:
def check_model_fit(df_eval_train,df_eval_test):
    df_combined = pd.concat([df_eval_train,df_eval_test],ignore_index=True)
    accuracy_train = df_eval_train['Accuracy'].values[0]
    accuracy_test = df_eval_test['Accuracy'].values[0]
    gap = accuracy_train - accuracy_test

    if accuracy_train < 0.60 or accuracy_test <0.60:
        status = 'Undeerfitting (Akurasi Rendah)'
    elif gap > 0.05:
        status = f'Overfitting (gap:{gap*100:.2f})'
    elif gap < -0.05:
        status = 'Overfitting / Data Leakage (Test > Train)'
    else:
        status = 'Good Fit'

    df_combined['Status'] = status
    return df_combined

In [9]:
df_eval_train = evaluate_model(y_pred_train,y_train,y_prob_train,evaluate='Training')
df_eval_test = evaluate_model(y_pred_test,y_test,y_prob_test,evaluate='Test')
df_eval = check_model_fit(df_eval_train,df_eval_test)

print('================================= PREDIKSI DENGAN SAMPLE DATASET ===================================')
print(df_eval.to_string())
print("\n" + "="*100 + "\n")

================================= PREDIKSI DENGAN SAMPLE DATASET ===================================
           Model Evaluated  Accuracy  Precision    Recall  F1-Score   ROC-AUC  Log Loss    Status
0  Decision Tree  Training  0.622879   0.014211  0.863889  0.027961  0.814110  0.577081  Good Fit
1  Decision Tree      Test  0.620509   0.013990  0.855556  0.027529  0.811427  0.578649  Good Fit




In [10]:
data_pasien = {
    'gender': ['F', 'F', 'F', 'F', 'M', 'M', 'F', 'M', 'F', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'F'],
    'age': [31, 72, 28, 63, 45, 68, 54, 39, 62, 48, 71, 58, 41, 65, 35, 52, 60, 47, 56, 50],
    'body_mass_index': [25.9, 28.6, 27.0, 27.0, 24.5, 31.2, 29.1, 22.8, 33.4, 26.3, 30.0, 28.1, 23.9, 32.5, 21.4, 27.8, 29.5, 25.0, 28.8, 26.7],
    'smoker': [0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1],
    'systolic_blood_pressure': [112.0, 125.0, 102.0, 130.0, 118.0, 145.0, 135.0, 110.0, 140.0, 122.0, 155.0, 128.0, 115.0, 150.0, 108.0, 124.0, 138.0, 120.0, 132.0, 126.0],
    'hypertension_treated': [1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0],
    'family_history_of_cardiovascular_disease': [0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0],
    'atrial_fibrillation': [0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    'chronic_kidney_disease': [0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0],
    'rheumatoid_arthritis': [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    'diabetes': [0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0],
    'chronic_obstructive_pulmonary_disorder': [0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0],
    'forced_expiratory_volume_1': [90.96, 88.00, 98.37, 100.00, 75.50, 82.40, 85.10, 99.00, 78.30, 92.00, 70.15, 88.50, 95.20, 81.00, 98.00, 91.40, 84.60, 93.00, 87.25, 89.90],
    'time_to_event_or_censoring': [9833, 100, 6954, 100, 5420, 120, 3110, 8740, 95, 4320, 150, 7230, 8900, 85, 9120, 6150, 240, 7540, 4890, 5100],
    'heart_attack': [1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1]
}
data_pasien_baru_x = pd.DataFrame(data_pasien)
data_pasien_baru_y = data_pasien_baru_x['heart_attack']

In [11]:
print("=== Melakukan Prediksi Data Pasien Baru ===")
prediksi_pasien = best_model_pipeline.predict(data_pasien_baru_x)
probabilitas_pasien = best_model_pipeline.predict_proba(data_pasien_baru_x)

hasil_df = data_pasien_baru_x.copy()
hasil_df['Prediksi Serangan Jantung'] = prediksi_pasien
hasil_df['Peluang Aman(%)'] = probabilitas_pasien[:,0] * 100
hasil_df['Peluang Terkena (%)'] = probabilitas_pasien[:,1] * 100

akurasi_bawaan = best_model_pipeline.score(data_pasien_baru_x, data_pasien_baru_y)
df_eval_data_baru = evaluate_model(
    pred=prediksi_pasien, 
    test=data_pasien_baru_y, 
    prob=probabilitas_pasien, 
    evaluate='Data_Baru'
)

print(f"Akurasi Model: {akurasi_bawaan * 100:.2f}%")
print("\nTabel Skor Evaluasi Lengkap Data Baru:")
print(df_eval_data_baru.to_string(index=False))
hasil_df[['Prediksi Serangan Jantung', 'Peluang Aman(%)', 'Peluang Terkena (%)']]

=== Melakukan Prediksi Data Pasien Baru ===
Akurasi Model: 15.00%

Tabel Skor Evaluasi Lengkap Data Baru:
        Model Evaluated  Accuracy  Precision   Recall  F1-Score  ROC-AUC  Log Loss
Decision Tree Data_Baru      0.15        0.1 0.111111  0.105263 0.181818  0.910831


,Prediksi Serangan Jantung,Peluang Aman(%),Peluang Terkena (%)
0,0,77.232204,22.767796
1,1,38.101160,61.898840
2,0,77.176913,22.823087
3,1,38.739833,61.260167
4,0,51.979024,48.020976
5,1,31.982874,68.017126
6,1,37.977513,62.022487
7,0,73.651036,26.348964
8,1,33.347410,66.652590
9,0,51.979024,48.020976


# 2.Multiclass Classification Random Forest